In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [4]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [5]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [6]:
import plotly.graph_objects as go


x_vals = road_positions[:, 0]
z_vals = road_positions[:, 2] 
indices = list(range(len(road_positions)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals, y=z_vals,
    mode='markers',
    marker=dict(size=5, color='blue'),
    text=indices,
    hovertemplate='<b>Index: %{text}</b><br>X: %{x:.1f}<br>Z: %{y:.1f}<extra></extra>' # 라벨도 Z로 표기
))

fig.update_layout(
    title="도로 점 확인용 지도 (X - Z 평면)",
    xaxis_title="X Axis (East/West)",
    yaxis_title="y Axis (North/South)", # Y축 라벨을 Z축으로 변경
    width=1000, height=800,
    hovermode='closest'
)

fig.show()

1 : Rx(수신기) 1개
2 : Rx 포트(편파 VH라서 2개)
3 : Tx 3개
128 : Tx 포트(8×8=64 소자 × VH 2편파 = 128)
9 : 경로 개수(현재 프레임에서 찾은 multipath 개수)
1 : time step(지금은 스냅샷 1개)

In [ ]:
frame_idx = 0  # 보고 싶은 프레임 인덱스(원하는 값으로 바꿔)
t = time_steps[frame_idx]
current_pos = get_pos_at_time(t)

# 위치/방향 업데이트
rx.position = current_pos
for name in tx_names:
    scene.transmitters[name].look_at(current_pos)

# 경로 계산
paths = solver(scene, max_depth=3, samples_per_src=100000,
               diffuse_reflection=True, diffraction=True,synthetic_array=True)

# CIR 추출
a, tau = paths.cir(out_type="tf", normalize_delays=False)

print("t =", t)
print("a shape:", a.shape)
print("tau shape:", tau.shape)


t = 0.0
a shape: (1, 2, 3, 128, 6, 1)
tau shape: (1, 3, 6)


In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# --------- Sionna 상수/타입(환경별 import 경로가 다를 수 있어 try 처리) ----------
try:
    from sionna.rt.constants import InteractionType, INVALID_SHAPE
except Exception:
    # fallback: InteractionType는 sionna.rt에서 가져오는 경우도 있음
    from sionna.rt import InteractionType
    INVALID_SHAPE = -1  # fallback (대부분 -1이 invalid로 쓰임)

TYPE_NAME = {
    int(getattr(InteractionType, "NONE", 0)): "None",
    int(getattr(InteractionType, "SPECULAR", 1)): "Specular",
    int(getattr(InteractionType, "DIFFUSE", 2)): "Diffuse",
    int(getattr(InteractionType, "REFRACTION", 3)): "Refraction",
    int(getattr(InteractionType, "DIFFRACTION", 4)): "Diffraction",
}

C = 299_792_458.0  # m/s

# --------- 캐시: 프레임별 paths를 저장 (Tx 바꿔도 재사용 가능) ----------
_paths_cache = {}  # frame_idx -> (t, pos, paths)

out = widgets.Output()

tx_dropdown = widgets.Dropdown(
    options=[("Tx_1", 0), ("Tx_2", 1), ("Tx_3", 2)],
    value=0,
    description="TX:",
    layout=widgets.Layout(width="180px")
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_steps)-1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="600px")
)

rel_delay_chk = widgets.Checkbox(
    value=False,
    description="Relative delay (min τ = 0)",
    indent=False
)

show_preview_chk = widgets.Checkbox(
    value=True,
    description="Show preview",
    indent=False
)

def _compute_paths_for_frame(frame_idx: int):
    if frame_idx in _paths_cache:
        return _paths_cache[frame_idx]

    t = float(time_steps[frame_idx])
    pos = get_pos_at_time(t)

    # Rx 위치 갱신
    rx.position = pos

    # (선택) Tx가 Rx를 바라보게
    for name in tx_names:
        scene.transmitters[name].look_at(pos)

    # paths 계산 (여기서 3개 Tx -> 1개 Rx 전체 링크 결과가 paths에 들어감)
    paths = solver(
        scene,
        max_depth=3,
        samples_per_src=100000,
        diffuse_reflection=True,
        diffraction=True,
        synthetic_array=True  # 너가 쓰던 설정 유지
    )

    _paths_cache[frame_idx] = (t, pos, paths)
    return _paths_cache[frame_idx]

def _slice_meta(paths, tx_idx: int):
    """
    interactions/vertices/objects/tau를 tx_idx에 대해 슬라이스해서
    [max_depth, P], [max_depth, P, 3], [max_depth, P], [P] 형태로 맞춰준다.
    synthetic_array 여부에 따라 인덱싱이 달라질 수 있어 try/fallback 처리.
    """
    syn = bool(getattr(paths, "synthetic_array", True))

    # tau
    tau_all = paths.tau
    # interactions / vertices / objects
    inter_all = getattr(paths, "interactions", None)
    verts_all = getattr(paths, "vertices", None)
    objs_all  = getattr(paths, "objects", None)

    if inter_all is None or verts_all is None or objs_all is None:
        raise AttributeError("paths에 interactions/vertices/objects가 없습니다. (버전/설정 확인 필요)")

    # 텐서를 numpy로
    tau_np   = tau_all.numpy()
    inter_np = inter_all.numpy()
    verts_np = verts_all.numpy()
    objs_np  = objs_all.numpy()

    # ----- synthetic array인 경우 (너의 tau shape: (1, 3, P)) -----
    if syn and tau_np.ndim == 3:
        # tau: [rx, tx, P]
        tau_tx = tau_np[0, tx_idx, :]

        # interactions: 보통 [max_depth, rx, tx, P]
        # vertices:     보통 [max_depth, rx, tx, P, 3]
        # objects:      보통 [max_depth, rx, tx, P]
        inter_tx = inter_np[:, 0, tx_idx, :]
        verts_tx = verts_np[:, 0, tx_idx, :, :]
        objs_tx  = objs_np[:, 0, tx_idx, :]

        return tau_tx, inter_tx, verts_tx, objs_tx

    # ----- non-synthetic(또는 다른 shape) fallback -----
    # 대표 포트(0,0) 기준으로 꺼내기
    # tau: [rx, rx_ant, tx, tx_ant, P]
    # interactions: [max_depth, rx, rx_ant, tx, tx_ant, P]
    # vertices:     [max_depth, rx, rx_ant, tx, tx_ant, P, 3]
    # objects:      [max_depth, rx, rx_ant, tx, tx_ant, P]
    tau_tx = tau_np[0, 0, tx_idx, 0, :]
    inter_tx = inter_np[:, 0, 0, tx_idx, 0, :]
    verts_tx = verts_np[:, 0, 0, tx_idx, 0, :, :]
    objs_tx  = objs_np[:, 0, 0, tx_idx, 0, :]

    return tau_tx, inter_tx, verts_tx, objs_tx

def _build_df(paths, tx_idx: int):
    # (1) CIR에서 a,tau 꺼내서 pdp_db/pathloss_db 만들기
    a, tau = paths.cir(out_type="tf", normalize_delays=False)

    tau_tx_tf = tau[0, tx_idx, :]  # (P,)
    pdp_tf = tf.reduce_sum(tf.abs(a[0, :, tx_idx, :, :, 0])**2, axis=[0, 1])  # (P,)

    # padding/무효 경로 제거 (tau<0 제거)
    valid = tf.math.is_finite(tau_tx_tf) & (tau_tx_tf >= 0)
    idx_valid = tf.where(valid)[:, 0]  # 원래 path_idx

    if tf.size(idx_valid) == 0:
        return pd.DataFrame(columns=[
            "arrival_rank","path_idx","tau_abs_ns","tau_rel_ns",
            "path_len_m","excess_len_m","pdp_db","pathloss_db",
            "interaction_seq","objects","vertices"
        ])

    tau_abs = tf.boolean_mask(tau_tx_tf, valid)      # (P_valid,)
    pdp_v   = tf.boolean_mask(pdp_tf, valid)         # (P_valid,)

    # 도착 순서(지연 순) 기준 정렬
    order = tf.argsort(tau_abs)
    tau_abs_s = tf.gather(tau_abs, order).numpy()    # seconds
    pdp_s     = tf.gather(pdp_v, order).numpy()
    path_idx_sorted = tf.gather(idx_valid, order).numpy().astype(int)

    tau_abs_ns = tau_abs_s * 1e9
    tau_rel_s  = tau_abs_s - np.min(tau_abs_s)
    tau_rel_ns = tau_rel_s * 1e9

    path_len_m   = C * tau_abs_s
    excess_len_m = C * tau_rel_s

    pdp_db = 10*np.log10(pdp_s + 1e-30)
    pathloss_db = -pdp_db  # 같은 정보(부호만 반대)

    # (2) paths 메타데이터(interactions/vertices/objects)로 경로 설명 붙이기
    tau_meta, inter_tx, verts_tx, objs_tx = _slice_meta(paths, tx_idx)

    # object_id -> name 매핑
    id2name = {}
    for name, obj in scene.objects.items():
        oid = getattr(obj, "object_id", None)
        if oid is not None:
            id2name[int(oid)] = name

    rows = []
    max_depth = inter_tx.shape[0]

    # 정렬된 path_idx 순서대로 메타 붙이기
    for rank, p in enumerate(path_idx_sorted):
        # interaction sequence
        codes = [int(inter_tx[k, p]) for k in range(max_depth)]
        codes = [c for c in codes if c != 0]  # NONE 제거
        seq = "LoS" if len(codes) == 0 else "->".join(TYPE_NAME.get(c, str(c)) for c in codes)

        # vertices: (0,0,0)은 무효로 보고 제거
        vlist = []
        for k in range(max_depth):
            pt = verts_tx[k, p, :]
            if not np.allclose(pt, 0):
                vlist.append(tuple(np.round(pt.astype(float), 3)))

        # objects: INVALID_SHAPE 제외
        olist = []
        for k in range(max_depth):
            oid = int(objs_tx[k, p])
            if oid != int(INVALID_SHAPE):
                olist.append(id2name.get(oid, f"id:{oid}"))

        rows.append({
            "arrival_rank": rank,
            "path_idx": int(p),
            "tau_abs_ns": float(tau_abs_ns[rank]),
            "tau_rel_ns": float(tau_rel_ns[rank]),
            "path_len_m": float(path_len_m[rank]),
            "excess_len_m": float(excess_len_m[rank]),
            "pdp_db": float(pdp_db[rank]),
            "pathloss_db": float(pathloss_db[rank]),
            "interaction_seq": seq,
            "objects": olist,
            "vertices": vlist
        })

    df = pd.DataFrame(rows)

    # 보기 좋게 반올림
    for c in ["tau_abs_ns","tau_rel_ns","path_len_m","excess_len_m","pdp_db","pathloss_db"]:
        df[c] = df[c].round(3)

    return df

def _update(_=None):
    frame_idx = frame_slider.value
    tx_idx = tx_dropdown.value
    rel_delay = rel_delay_chk.value
    show_preview = show_preview_chk.value

    with out:
        out.clear_output(wait=True)

        t, pos, paths = _compute_paths_for_frame(frame_idx)

        # preview
        if show_preview:
            scene.preview(paths=paths, show_devices=True, resolution=[800, 600])
            print(f"t={t:.2f}s | frame={frame_idx} | pos={pos}")

        # df (path_idx ↔ 의미)
        df = _build_df(paths, tx_idx)
        if df.empty:
            print(f"Tx_{tx_idx+1}: 유효 경로가 없습니다.")
            return

        display(df)

        # CIR stem (x축은 절대/상대 선택)
        x = df["tau_rel_ns"].values if rel_delay else df["tau_abs_ns"].values
        xlabel = "Delay τ (ns, relative)" if rel_delay else "Delay τ (ns, absolute)"

        plt.figure(figsize=(8, 3.6))
        plt.stem(x, df["pdp_db"].values, basefmt=" ")
        for xi, yi, pid in zip(x, df["pdp_db"].values, df["path_idx"].values):
            plt.text(xi, yi, str(pid), fontsize=9, ha="center", va="bottom")
        plt.xlabel(xlabel)
        plt.ylabel("PDP (dB)  = 10log10(Σ|a|²)")
        plt.title(f"Tx_{tx_idx+1} | t={t:.2f}s | frame={frame_idx}")
        plt.grid(True)
        plt.show()

# 이벤트 연결
frame_slider.observe(_update, names="value")
tx_dropdown.observe(_update, names="value")
rel_delay_chk.observe(_update, names="value")
show_preview_chk.observe(_update, names="value")

display(widgets.VBox([
    widgets.HBox([frame_slider, tx_dropdown]),
    widgets.HBox([rel_delay_chk, show_preview_chk]),
    out
]))

_update()